### ЗАДАЧА: Реестр абонементов фитнес-клуба

Администратор фитнес-клуба получает строки с данными об абонементах.
Нужно собрать удобную модель, которая позволит:
- загрузить клиентов в единый реестр,
- посмотреть только активные абонементы,
- отфильтровать клиентов по тарифу,
- посчитать суммарное число оставшихся посещений,
- понять, как меняется реестр после активации и списания посещения.

В данных есть абонементы с разными статусами и остатком посещений,
поэтому важно корректно валидировать тариф, статус и изменение состояния объекта.


In [44]:
# rows: sub_id|client_name|plan|visits_left|status
rows = [
    'SB-100|Alice|standard|8|active',
    'SB-101|Bob|premium|12|frozen',
    'SB-102|Charlie|family|0|expired',
    'SB-103|Diana|standard|5|active',
]


class Subscription:
    allowed_plans = {'standard', 'premium', 'family'}
    allowed_statuses = {'active', 'frozen', 'expired'}

    def __init__(self, sub_id, client_name, plan, visits_left, status):
        # TODO: сохранить sub_id, client_name, plan
        self.sub_id = sub_id
        self.client_name = client_name
        self.plan = plan
        # TODO: visits_left хранить через self._visits_left
        self.visits_left = visits_left
        # TODO: значение visits_left пропустить через property/setter
        # TODO: проверить plan и status, иначе raise ValueError
        if plan not in self.allowed_plans:
            raise ValueError("Неверный план")
        if status not in self.allowed_statuses:
            raise ValueError("Неверный статус")
        self.status = status

    @property
    def visits_left(self):
        # TODO: вернуть текущее число посещений
        return self._visits_left

    @visits_left.setter
    def visits_left(self, value):
        # TODO: привести value к int
        try:
            value = int(value)
        except:
            raise ValueError("Количиство визитов должно быть числом")
        # TODO: если value < 0 -> raise ValueError('Visits must be >= 0')
        if value < 0:
            raise ValueError("Количестов визитов не может быть отрицательным")
        # TODO: сохранить результат в self._visits_left
        self._visits_left = value

    def use_visit(self):
        # TODO: если статус не 'active' -> raise ValueError
        if self.status != "active":
            raise ValueError("Неподходящий статус")
        # TODO: если visits_left == 0 -> raise ValueError
        if self._visits_left == 0:
            raise ValueError("Не осталось количества визитов")
        # TODO: уменьшить visits_left на 1
        self._visits_left -= 1
        # TODO: если после списания visits_left == 0, перевести статус в 'expired'
        if self._visits_left == 0:
            self.status = "expired"

    def freeze(self):
        # TODO: если статус 'expired' -> raise ValueError
        if self.status == "expired":
            raise ValueError("Неподходящий статус")
        # TODO: перевести абонемент в 'frozen'
        self.status = "frozen"
        

    def activate(self):
        # TODO: если visits_left == 0 -> raise ValueError
        if self._visits_left == 0:
            raise ValueError("Не осталось количества визитов")
        # TODO: перевести абонемент в 'active'
        self.status = "active"

    @classmethod
    def from_row(cls, row):
        # TODO: split по '|'
        parts = row.split("|")
        # TODO: ожидать 5 частей: sub_id, client_name, plan, visits_left, status
        if len(parts) != 5:
            raise ValueError("Количество частей должно состоять из 5")
        sub_id, client_name, plan, _visits_left, status = parts
        # TODO: вернуть Subscription(...)
        return Subscription(sub_id, client_name, plan, _visits_left, status)

    def __repr__(self):
        # TODO: вернуть строку вида Subscription(sub_id='...', client_name='...', status='...')
        return f"Subscription(sub_id='{self.sub_id}', client_name='{self.client_name}', status='{self.status}')"


class SubscriptionRegistry:
    def __init__(self):
        self.items = []

    def add(self, subscription):
        # TODO: добавить subscription в self.items
        self.items.append(subscription)

    def load(self, rows):
        # TODO: для каждой строки создать Subscription.from_row(row)
        for row in rows:
            subscription = Subscription.from_row(row)
        # TODO: добавить объект в реестр через add(...)
            self.add(subscription)

    def active_subscriptions(self):
        # TODO: вернуть список абонементов со статусом 'active'
        return [item for item in self.items if item.status == "active"]

    def by_plan(self, plan):
        # TODO: вернуть список абонементов нужного тарифа
        return [item for item in self.items if item.plan == plan]

    def total_visits_left(self):
        # TODO: вернуть суммарное число оставшихся посещений
        return sum(item.visits_left for item in self.items)
    
    def status_summary(self):
        # TODO: собрать dict вида status -> count
        status_summary = {}
        for item in self.items:
            status_summary[item.status] = status_summary.get(item.status, 0)
            status_summary[item.status] += 1
        return status_summary
       
    def find(self, sub_id):
        # TODO: вернуть абонемент по sub_id или None
        for item in self.items:
            if item.sub_id == sub_id:
                return item
        return None

registry = SubscriptionRegistry()

# TODO: загрузить rows в registry
registry.load(rows)
# TODO: вывести все абонементы
print("Все абонементы:")
for item in registry.items:
    print(item)
# TODO: вывести active_subscriptions()
print("Все активные абонементы:")
for item in registry.active_subscriptions():
    print(item)
# TODO: вывести by_plan('standard')
print("Активные абонементы с тарифным планом 'standard':")
for item in registry.by_plan('standard'):
    print(item)
# TODO: вывести total_visits_left()
print(f"Суммарное количество оставшихся посещений = {registry.total_visits_left()} раз.")
# TODO: вывести status_summary()
status_summary = registry.status_summary()
print("Суммарное количество статусов:")
for status, count in status_summary.items():
    print(f"Статус '{status}' встречается {count} раз")
# TODO: найти абонемент 'SB-101', активировать его и вывести status_summary()\
find_101 = registry.find("SB-101")
if find_101:
    find_101.activate()
else:
    print("Такой id не найден")
status_summary_new = registry.status_summary()
print("После активации id: SB-101, количество статусов:")
for status, count in status_summary_new.items():
    print(f"Статус '{status}' встречается {count} раз")
# TODO: найти абонемент 'SB-100', списать одно посещение и вывести объект
find_100 = registry.find("SB-100")
if find_100:
    find_100.use_visit()
else:
    print("Такой id не найден")
print(f"После списания у {find_100} осталось {find_100.visits_left} посещений.")

Все абонементы:
Subscription(sub_id='SB-100', client_name='Alice', status='active')
Subscription(sub_id='SB-101', client_name='Bob', status='frozen')
Subscription(sub_id='SB-102', client_name='Charlie', status='expired')
Subscription(sub_id='SB-103', client_name='Diana', status='active')
Все активные абонементы:
Subscription(sub_id='SB-100', client_name='Alice', status='active')
Subscription(sub_id='SB-103', client_name='Diana', status='active')
Активные абонементы с тарифным планом 'standard':
Subscription(sub_id='SB-100', client_name='Alice', status='active')
Subscription(sub_id='SB-103', client_name='Diana', status='active')
Суммарное количество оставшихся посещений = 25 раз.
Суммарное количество статусов:
Статус 'active' встречается 2 раз
Статус 'frozen' встречается 1 раз
Статус 'expired' встречается 1 раз
После активации id: SB-101, количество статусов:
Статус 'active' встречается 3 раз
Статус 'expired' встречается 1 раз
После списания у Subscription(sub_id='SB-100', client_name='